# 타이타닉 생존자 분석 — 룰베이스 분류 도출 (실습)

## 분석 시나리오
1912년 4월, 타이타닉호가 빙산과 충돌해 침몰했다. 891명 승객의 인적사항(나이, 성별, 객실등급, 요금, 호칭 등)과 생존 여부 데이터가 있다.
어떤 사람들이 생존했는지 패턴을 찾고, 단순한 룰만으로 생존 여부를 예측할 수 있는지 확인해본다.

## 분석 흐름 
1. **데이터 파악** — 컬럼, 결측치, 분포 
2. **시각화 탐색** — 변수별 생존율 비교 
3. **호칭 파생** — 이름에서 호칭 추출, 호칭별 생존율 
4. **인사이트 정리** — 어떤 변수가 생존을 좌우하는가 
5. **룰 도출 및 검증** — 룰 정의 후 정확도/recall 평가

## 데이터
- 파일: `타이타닉_데이터.csv` (891행, 11컬럼)
- 타겟: `생존여부` (`생존` / `사망`)

> **참고:** 본 데이터는 캐글 Titanic Competition의 `train.csv`를 한국어로 가공한 것이다.

---

## 0. 환경 준비

- 다음을 실행하여 환경을 준비하시오.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 자동 감지
candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic',
              'Noto Sans CJK KR', 'Noto Sans KR', 'DejaVu Sans']
available = {f.name for f in fm.fontManager.ttflist}
for font in candidates:
    if font in available:
        plt.rcParams['font.family'] = font
        break
plt.rcParams['axes.unicode_minus'] = False

sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
pd.set_option('display.max_columns', None)

- 다음 프롬프트로 AI를 단계별 데이터 분석 도우미로 세팅하시오.

```
# 역할
당신은 유능한 15년차 데이터 분석가입니다.
주어진 데이터에 대해서 공학 박사급 통찰력을 발휘하여 데이터 분석 코드를 작성하세요.

# 환경
데이터를 당신에게 모두 업로드 할 수 없는 환경입니다. 
데이터 샘플 100행 정도를 첨부하면 전체 데이터의 스키마를 파악하고 사용자 질문에 대응하세요.
한글 출력 환경이 세팅되어 있기 때문에 sns.set_style("whitegrid") 같이 따로 스타일을 초기화 하지 마세요.

# 응답
친절하고 상대방을 존중하는 톤으로 이야기합니다. 늘 충실한 데이터 분석가 동료입장을 유지하세요.
너무 많은 정보를 한꺼번에 주지 말고 단계적으로 사용자의 요구에 응답하세요.

# 분석의 범위
EDA를 수행하고 의미있는 변수를 추출합니다.
이 결과로 부터 데이터의 인사이트를 넣고 의사결정에 도움이 될만한 룰을 발굴합니다.
ML 모델링은 범위에 포함되지 않습니다.
```

---
## 1. 데이터 파악 

### 과제 1-1. 데이터를 불러오고 모양과 처음 5행을 확인하시오.

**해석:** 

### 과제 1-2. 각 컬럼의 데이터 타입과 결측치를 확인하시오.

**해석:** 

### 과제 1-3. 수치형 변수의 기초 통계량을 확인하시오.

**해석:** 

### 과제 1-4. 타겟 변수 `생존여부`의 분포를 확인하시오.

**해석:** 

### 과제 1-5. 범주형 변수의 카테고리별 빈도를 확인하시오.

**해석:** 

---
## 2. 시각화 탐색 

이제 각 변수가 생존여부와 어떤 관계가 있는지 살펴본다.

### 과제 2-1. 수치형 변수들의 분포를 확인하시오.

**해석:**

### 과제 2-2. 수치형 변수 각각에 대해 `생존여부`별 박스플롯을 그려 비교하시오.

**해석:**

### 과제 2-3. 객실등급별 생존율을 확인하시오.

**해석:**

### 과제 2-4. 성별 생존율을 확인하시오.

**해석:**

### 과제 2-5. 나이 구간별 생존율을 확인하시오.

**해석:**

### 과제 2-6. 승선항별 생존율을 확인하시오.

**해석:** 

### 과제 2-7. 동승가족수와 생존율의 관계를 확인하시오.

**해석:**

---
## 3. 호칭 파생

`이름` 컬럼을 자세히 보면 흥미로운 정보가 들어있다. 예를 들어:
- `Braund, Mr. Owen Harris`
- `Heikkinen, Miss. Laina`
- `Allen, Mrs. William Henry`

쉼표 다음, 마침표 앞에 **호칭(Title)** 이 있다 — Mr, Mrs, Miss, Master 등. 호칭은 그 사람의 성별, 결혼상태, 사회적 지위를 한 글자로 압축해 보여주는 정보다.

**왜 호칭이 중요한가?**
- 단순히 성별만 보면 남성은 거의 다 사망한 것처럼 보이지만, **남자아이(Master)는 생존율이 다르다**
- Mrs(기혼 여성)와 Miss(미혼 여성)도 다를 수 있다
- Dr, Rev 등 직함 정보도 들어있다

**핵심 스킬:** 정규표현식(regex)으로 텍스트에서 패턴을 추출하는 작업이다. 빈칸 채우기 식 작업으로는 LLM 도움이 가장 빛을 발하는 영역이다.

### 과제 3-1. `이름` 컬럼에서 호칭을 추출해 새 컬럼 `호칭`에 저장하시오.



**해석:** 

### 과제 3-2. 호칭별 빈도를 확인하시오.

**해석:**

### 과제 3-3. 빈도가 적은 호칭들을 `Other`로 묶어 `호칭_그룹` 컬럼을 만드시오.

**해석:** 

### 과제 3-4. 호칭_그룹별 생존율을 확인하시오.

**해석:**

---
## 4. 인사이트 정리 

### 과제 4-1. 지금까지의 탐색 결과를 표로 정리하고, 어떤 변수가 생존을 결정하는지 평가하시오.

**해석:** 

---
## 5. 룰 도출 및 검증 

### 과제 5-1. 위 인사이트를 바탕으로 세 가지 룰을 정의하고, 각각의 정확도를 베이스라인과 비교하시오.

**해석:**

### 과제 5-2. 룰을 단계적으로 결합하여 최종 룰을 만들고, 각 단계의 성능을 비교하시오.

**해석:**

### 과제 5-3. 최종 룰(룰1+2+3)의 혼동행렬을 시각화하고 분류 리포트를 출력하시오.

**해석:**